In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/harshk1111/chartqa-dataset/ChartQA-Dataset/test/test_augmented.json
/kaggle/input/datasets/harshk1111/chartqa-dataset/ChartQA-Dataset/test/test_human.json
/kaggle/input/datasets/harshk1111/chartqa-dataset/ChartQA-Dataset/test/png/00339007006077.png
/kaggle/input/datasets/harshk1111/chartqa-dataset/ChartQA-Dataset/test/png/00795994017065.png
/kaggle/input/datasets/harshk1111/chartqa-dataset/ChartQA-Dataset/test/png/01001540004402.png
/kaggle/input/datasets/harshk1111/chartqa-dataset/ChartQA-Dataset/test/png/01499440003158.png
/kaggle/input/datasets/harshk1111/chartqa-dataset/ChartQA-Dataset/test/png/01729694006399.png
/kaggle/input/datasets/harshk1111/chartqa-dataset/ChartQA-Dataset/test/png/02267499005481.png
/kaggle/input/datasets/harshk1111/chartqa-dataset/ChartQA-Dataset/test/png/02534409005100.png
/kaggle/input/datasets/harshk1111/chartqa-dataset/ChartQA-Dataset/test/png/03672594001226.png
/kaggle/input/datasets/harshk1111/chartqa-dataset/ChartQA-Dataset/test

In [7]:
from datasets import load_dataset
dataset = load_dataset("Ahren09/InfoVQA")

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

In [8]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer', 'image', 'deck_name', 'image_url'],
        num_rows: 23946
    })
    val: Dataset({
        features: ['id', 'question', 'answer', 'image', 'deck_name', 'image_url'],
        num_rows: 2801
    })
    test: Dataset({
        features: ['id', 'question', 'answer', 'image', 'deck_name', 'image_url'],
        num_rows: 3288
    })
})


In [9]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.replace("/kaggle/input", "").count(os.sep)
    if depth <= 3:
        print(root, "|", len(files), "files |", files[:5])
        

/kaggle/input | 0 files | []
/kaggle/input/datasets | 0 files | []
/kaggle/input/datasets/harshk1111 | 0 files | []
/kaggle/input/datasets/harshk1111/chartqa-dataset | 0 files | []
/kaggle/input/datasets/abdul1hadi | 0 files | []
/kaggle/input/datasets/abdul1hadi/chartqa | 0 files | []
/kaggle/input/datasets/abdul1hadi/lgovf-code | 2 files | ['lg_ovf.py', 'lgovf_metrics.py']


In [10]:
import re
from PIL import Image
from datasets import DatasetDict

def clean_text(t):
    if t is None: return ""
    t = str(t).strip()
    t = re.sub(r"\s+", " ", t)
    return t

def norm_answer(a):
    a = clean_text(a).lower()
    a = a.rstrip(".")
    return a

def preprocess(ex):
    ex["question"] = clean_text(ex["question"])
    ex["answer"] = norm_answer(ex["answer"])
    img = ex["image"]
    if img is not None:
        img = img.convert("RGB")
        img.thumbnail((1024, 1024))      # keep aspect ratio, cap size
        ex["image"] = img
    return ex

def is_valid(ex):
    return bool(ex["question"]) and bool(ex["answer"]) and ex["image"] is not None

dataset = dataset.map(preprocess, num_proc=2)
dataset = dataset.filter(is_valid)

# drop duplicate (image_url, question) pairs within train
seen = set()
def dedup(ex):
    key = (ex["image_url"], ex["question"])
    if key in seen: return False
    seen.add(key); return True
dataset["train"] = dataset["train"].filter(dedup)

# keep only the columns you need, rename for the common schema
dataset = dataset.map(lambda ex: {"source": "slides"})
print(dataset)

Map (num_proc=2):   0%|          | 0/23946 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/2801 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/3288 [00:00<?, ? examples/s]

Filter:   0%|          | 0/23946 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2801 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3288 [00:00<?, ? examples/s]

Filter:   0%|          | 0/23946 [00:00<?, ? examples/s]

Map:   0%|          | 0/23939 [00:00<?, ? examples/s]

Map:   0%|          | 0/2801 [00:00<?, ? examples/s]

Map:   0%|          | 0/3288 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer', 'image', 'deck_name', 'image_url', 'source'],
        num_rows: 23939
    })
    val: Dataset({
        features: ['id', 'question', 'answer', 'image', 'deck_name', 'image_url', 'source'],
        num_rows: 2801
    })
    test: Dataset({
        features: ['id', 'question', 'answer', 'image', 'deck_name', 'image_url', 'source'],
        num_rows: 3288
    })
})


In [11]:
import json, glob, os, pandas as pd
from datasets import Dataset, DatasetDict, Image as HFImage

ROOT = "/kaggle/input/datasets/harshk1111/chartqa-dataset/ChartQA-Dataset"

def load_split(split):
    rows = []
    for f in sorted(glob.glob(f"{ROOT}/{split}/*.json")):
        sub = "human" if "human" in os.path.basename(f) else "augmented"
        for r in json.load(open(f)):
            r["subset"] = sub
            rows.append(r)
    assert rows, f"No JSON found in {ROOT}/{split}"

    # index png files once instead of globbing per row
    png = {os.path.basename(p): p for p in glob.glob(f"{ROOT}/{split}/png/*.png")}

    df = pd.DataFrame(rows).rename(columns={"query": "question", "label": "answer"})
    df["image"] = df["imgname"].map(png)
    df = df.dropna(subset=["image"])                      # drop rows whose image is missing
    df["answer"] = df["answer"].astype(str).map(norm_answer)
    df["question"] = df["question"].map(clean_text)
    df = df[(df.question != "") & (df.answer != "")]
    df = df.drop_duplicates(subset=["imgname", "question"]).reset_index(drop=True)
    df["id"] = df["imgname"] + "_" + df.index.astype(str)
    df["source"] = "chartqa"
    return Dataset.from_pandas(df[["id", "question", "answer", "image", "source", "subset"]],
                               preserve_index=False).cast_column("image", HFImage())

chartqa = DatasetDict({s: load_split(s) for s in ["train", "val", "test"]})
print(chartqa)
chartqa = chartqa.map(preprocess, num_proc=2)
print(chartqa)

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer', 'image', 'source', 'subset'],
        num_rows: 28066
    })
    val: Dataset({
        features: ['id', 'question', 'answer', 'image', 'source', 'subset'],
        num_rows: 1904
    })
    test: Dataset({
        features: ['id', 'question', 'answer', 'image', 'source', 'subset'],
        num_rows: 2490
    })
})


Map (num_proc=2):   0%|          | 0/28066 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/1904 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/2490 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer', 'image', 'source', 'subset'],
        num_rows: 28066
    })
    val: Dataset({
        features: ['id', 'question', 'answer', 'image', 'source', 'subset'],
        num_rows: 1904
    })
    test: Dataset({
        features: ['id', 'question', 'answer', 'image', 'source', 'subset'],
        num_rows: 2490
    })
})


In [12]:
print(chartqa)

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer', 'image', 'source', 'subset'],
        num_rows: 28066
    })
    val: Dataset({
        features: ['id', 'question', 'answer', 'image', 'source', 'subset'],
        num_rows: 1904
    })
    test: Dataset({
        features: ['id', 'question', 'answer', 'image', 'source', 'subset'],
        num_rows: 2490
    })
})


In [13]:
!find /kaggle/input -name "lg_ovf.py"

/kaggle/input/datasets/abdul1hadi/lgovf-code/lg_ovf.py


In [14]:
import sys
sys.path.append("/kaggle/input/datasets/abdul1hadi/lgovf-code")
from lg_ovf import *

In [15]:
!pip install -q transformers datasets sacrebleu rapidfuzz pytesseract scipy torchvision
!apt-get install -y -q tesseract-ocr     # only needed if a dataset has no OCR words/boxes

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (5.3.4-1build5).
0 upgraded, 0 newly installed, 0 to remove and 54 not upgraded.


In [16]:
!find /kaggle/input -maxdepth 4 -iname "*docvqa*" | head
!find /kaggle/input -name "*.arrow" 2>/dev/null | head -5

In [17]:
P = "/kaggle/input/notebooks/abdul1hadi/mp-dov-vqa"
!echo "--- sizes ---"; ls -lh $P/mp-docvqa.zip; du -sh $P/mp-docvqa
!echo "--- folder contents (ignoring hidden .cache) ---"
!find $P/mp-docvqa -not -path "*/.cache/*" | head -40
!echo "--- data files anywhere in the folder ---"
!find $P/mp-docvqa \( -name "*.parquet" -o -name "*.arrow" -o -name "*.json*" -o -name "*.csv" \) | head -20
!echo "--- zip contents ---"
!unzip -l $P/mp-docvqa.zip | head -30


--- sizes ---
ls: cannot access '/kaggle/input/notebooks/abdul1hadi/mp-dov-vqa/mp-docvqa.zip': No such file or directory
du: cannot access '/kaggle/input/notebooks/abdul1hadi/mp-dov-vqa/mp-docvqa': No such file or directory
--- folder contents (ignoring hidden .cache) ---
find: ‘/kaggle/input/notebooks/abdul1hadi/mp-dov-vqa/mp-docvqa’: No such file or directory
--- data files anywhere in the folder ---
find: ‘/kaggle/input/notebooks/abdul1hadi/mp-dov-vqa/mp-docvqa’: No such file or directory
--- zip contents ---
unzip:  cannot find or open /kaggle/input/notebooks/abdul1hadi/mp-dov-vqa/mp-docvqa.zip, /kaggle/input/notebooks/abdul1hadi/mp-dov-vqa/mp-docvqa.zip.zip or /kaggle/input/notebooks/abdul1hadi/mp-dov-vqa/mp-docvqa.zip.ZIP.


In [18]:
import os, glob, collections, pyarrow.parquet as pq
D = "/kaggle/input/notebooks/abdul1hadi/mp-dov-vqa/mp-docvqa/data"

print(collections.Counter(os.path.basename(f).split("-")[0] for f in glob.glob(D + "/*.parquet")))

f = sorted(glob.glob(D + "/val-*.parquet"))[0]
pf = pq.ParquetFile(f)
names = pf.schema_arrow.names
print(names)

want = [c for c in ["question", "answers", "answer_page_idx", "doc_id", "words_1", "boxes_1"] if c in names]
rows = pf.read_row_group(0, columns=want).slice(0, 2).to_pylist()
r = rows[0]
print({k: (v if k in ("question", "answers", "answer_page_idx", "doc_id") else str(v)[:150]) for k, v in r.items()})
if "boxes_1" in r and r["boxes_1"]:
    flat = [x for b in r["boxes_1"] for x in b]
    print("box value range:", min(flat), max(flat), "| n words:", len(r["words_1"]))

Counter()


IndexError: list index out of range

In [21]:
!apt-get -qq install tesseract-ocr
!pip -q install pytesseract sacrebleu

In [22]:
import os, re, json, math, random, hashlib
from dataclasses import dataclass, asdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
from transformers import (ViTModel, ViTConfig, BertModel, BertConfig, AutoTokenizer,
                          get_linear_schedule_with_warmup)

print("GPUs visible:", torch.cuda.device_count())   # must print 2
print("CPU cores   :", os.cpu_count())              # should print 4

@dataclass
class Cfg:
    vit_name: str = "google/vit-base-patch16-224-in21k"
    bert_name: str = "bert-base-uncased"
    img_size: int = 512
    max_text_len: int = 512
    max_ans_len: int = 32
    batch_size: int = 16           # split across 2 GPUs = 8 each
    grad_accum: int = 2            # 16 x 2 = effective 32
    epochs: int = 5                # <-- changed
    patience: int = 2              # <-- changed
    lr_pre: float = 3e-5
    lr_new: float = 1e-4
    weight_decay: float = 0.01
    warmup_frac: float = 0.10
    clip: float = 1.0
    dropout: float = 0.1
    max_val: int = 1500
    num_workers: int = 4           # <-- changed: 4 CPU cores
    fp16: bool = True
    seed: int = 42
    out_dir: str = "/kaggle/working/ckpt"

GPUs visible: 2
CPU cores   : 4


In [23]:
import os
os.environ["OMP_THREAD_LIMIT"] = "1"        # one thread per Tesseract process
from datasets import DatasetDict

def add_ocr_fast(dsdict, col, key_fn, num_proc=4):
    """OCR each unique image once, then copy words/boxes to every question that uses it.
    col    : metadata column that identifies the image
    key_fn : maps that column's value to a unique image key"""
    import pytesseract
    out = {}
    for split, ds in dsdict.items():
        keys = [key_fn(x) for x in ds[col]]
        first = {}
        for i, k in enumerate(keys): first.setdefault(k, i)
        uniq = ds.select(sorted(first.values()))
        print(f"{split}: {len(ds)} questions -> {len(uniq)} unique images")

        def _ocr(ex):
            img = ex["image"].convert("RGB")
            d = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)
            W, H = img.size; words, boxes = [], []
            for t, x, y, w, h, c in zip(d["text"], d["left"], d["top"], d["width"], d["height"], d["conf"]):
                t = t.strip()
                if not t or float(c) < 0: continue
                b = [1000*x/W, 1000*y/H, 1000*(x+w)/W, 1000*(y+h)/H]
                boxes.append([int(min(max(v, 0), 1000)) for v in b]); words.append(t)
            return {"words": words, "boxes": boxes}

        drop = [c for c in uniq.column_names if c != col]          # don't rewrite images
        o = uniq.map(_ocr, num_proc=num_proc, remove_columns=drop)
        table = {key_fn(c): (w, b) for c, w, b in zip(o[col], o["words"], o["boxes"])}
        out[split] = ds.map(lambda c: {"words": table[key_fn(c)][0], "boxes": table[key_fn(c)][1]},
                            input_columns=[col])
    return DatasetDict(out)

# ChartQA: image name = id without the trailing "_<index>"
chart  = add_ocr_fast(chartqa, "id", lambda s: s.rsplit("_", 1)[0], num_proc=4)
# Slides: image identified by its URL column (run dataset["train"].column_names to confirm)
slides = add_ocr_fast(dataset, "image_url", lambda s: s, num_proc=4)

chart.save_to_disk("/kaggle/working/chart_ocr")
slides.save_to_disk("/kaggle/working/slides_ocr")

train: 28066 questions -> 18316 unique images


Map (num_proc=4):   0%|          | 0/18316 [00:00<?, ? examples/s]

Map:   0%|          | 0/28066 [00:00<?, ? examples/s]

val: 1904 questions -> 1054 unique images


Map (num_proc=4):   0%|          | 0/1054 [00:00<?, ? examples/s]

Map:   0%|          | 0/1904 [00:00<?, ? examples/s]

... (14 outputs left out)

Saving the dataset (0/1 shards):   0%|          | 0/2490 [00:00<?, ? examples/s]

Saving the dataset (0/21 shards):   0%|          | 0/23939 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/2801 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/3288 [00:00<?, ? examples/s]

In [24]:
class DS(Dataset):
    def __init__(self, hf): self.hf = hf
    def __len__(self): return len(self.hf)
    def __getitem__(self, i): return self.hf[i]

class Collator:
    def __init__(self, tok, cfg, train):
        self.tok, self.cfg, self.train = tok, cfg, train
        self.aug = T.Compose([T.ColorJitter(0.2, 0.2), T.RandomRotation(2, fill=255)]) if train else None
        self.norm = T.Normalize([0.5]*3, [0.5]*3)

    def prep_image(self, img):
        S = self.cfg.img_size
        img = img.convert("RGB"); w, h = img.size; r = S / max(w, h)
        img = img.resize((max(1, int(w*r)), max(1, int(h*r))), Image.BICUBIC)
        canvas = Image.new("RGB", (S, S), (255, 255, 255)); canvas.paste(img, (0, 0))
        if self.aug: canvas = self.aug(canvas)
        x = self.norm(T.functional.to_tensor(canvas))
        if self.train: x = x + 0.02 * torch.randn_like(x)
        return x

    def __call__(self, batch):
        cfg, tok = self.cfg, self.tok
        pix, ids, att, tt, bxs, dec = [], [], [], [], [], []
        for ex in batch:
            pix.append(self.prep_image(ex["image"]))
            words = list(ex.get("words") or []); boxes = list(ex.get("boxes") or [])
            if not words: words, boxes = ["."], [[0, 0, 0, 0]]
            q = ex["question"].split() or ["?"]
            enc = tok(q, words, is_split_into_words=True, truncation="only_second", max_length=cfg.max_text_len)
            wid, sid = enc.word_ids(), enc.sequence_ids()
            bb = []
            for k, (w, s) in enumerate(zip(wid, sid)):
                if s is None: bb.append([0]*4 if k == 0 else [1000]*4)
                elif s == 0: bb.append([0]*4)
                else: bb.append(boxes[w])
            ids.append(torch.tensor(enc["input_ids"])); att.append(torch.tensor(enc["attention_mask"]))
            tt.append(torch.tensor(enc["token_type_ids"])); bxs.append(torch.tensor(bb))
            a = tok(ex["answer"], truncation=True, max_length=cfg.max_ans_len)["input_ids"]
            dec.append(torch.tensor(a))
        pad = nn.utils.rnn.pad_sequence
        return dict(pixel=torch.stack(pix), ids=pad(ids, True, tok.pad_token_id), att=pad(att, True, 0),
                    tt=pad(tt, True, 0), boxes=pad(bxs, True, 0), dec=pad(dec, True, tok.pad_token_id),
                    answers=[ex["answer"] for ex in batch])

In [27]:
class LayoutEmbedding(nn.Module):
    def __init__(self, d):
        super().__init__(); self.t = nn.ModuleList([nn.Embedding(1001, d) for _ in range(4)])
    def forward(self, b): return sum(self.t[i](b[..., i].clamp(0, 1000)) for i in range(4))

class CrossAttnFusion(nn.Module):
    def __init__(self, d, heads, p):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, heads, dropout=p, batch_first=True)
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Dropout(p), nn.Linear(4*d, d))
        self.drop = nn.Dropout(p)
    def forward(self, Tk, L, V):
        a, _ = self.attn(Tk + L, V, V)
        x = self.n1(Tk + self.drop(a))
        return self.n2(x + self.drop(self.ffn(x)))

class LGOVF(nn.Module):
    def __init__(self, variant, cfg):
        super().__init__()
        assert variant in {"v1", "v2", "v3", "v4", "v5"}
        self.variant, self.cfg = variant, cfg
        self.vit = ViTModel.from_pretrained(cfg.vit_name, add_pooling_layer=False)
        self.bert = BertModel.from_pretrained(cfg.bert_name, add_pooling_layer=False)
        d = self.bert.config.hidden_size; heads = d // 64; p = cfg.dropout
        V = self.bert.config.vocab_size
        self.layout = LayoutEmbedding(d)
        self.type_emb = nn.Embedding(2, d)
        self.fuse_proj = nn.Linear(2*d, d)
        self.xfuse = CrossAttnFusion(d, heads, p)
        self.mem_norm = nn.LayerNorm(d)
        self.tok_emb = nn.Embedding(V, d)
        self.tok_emb.weight.data.copy_(self.bert.embeddings.word_embeddings.weight.data)
        self.pos_emb = nn.Embedding(cfg.max_ans_len + 2, d)
        layer = nn.TransformerDecoderLayer(d, heads, 4*d, p, activation="gelu", batch_first=True, norm_first=True)
        self.decoder = nn.TransformerDecoder(layer, 2)
        self.dec_norm = nn.LayerNorm(d)

    def encode(self, pixel, ids, att, tt, boxes):
        v_ = self.variant
        t_att = att * (tt == 0) if v_ == "v1" else att
        B = ids.size(0)
        if v_ != "v2":
            Vf = self.vit(pixel_values=pixel, interpolate_pos_encoding=True).last_hidden_state
        Tk = self.bert(input_ids=ids, attention_mask=t_att, token_type_ids=tt).last_hidden_state
        tmask = t_att == 0
        if v_ == "v2":
            mem, m = Tk, tmask
        elif v_ == "v5":
            mem, m = self.xfuse(Tk, self.layout(boxes), Vf), tmask
        else:
            if v_ == "v4": Tk = self.fuse_proj(torch.cat([Tk, self.layout(boxes)], -1))
            Vf = Vf + self.type_emb.weight[0]; Tk = Tk + self.type_emb.weight[1]
            mem = torch.cat([Vf, Tk], 1)
            m = torch.cat([torch.zeros(B, Vf.size(1), dtype=torch.bool, device=ids.device), tmask], 1)
        return self.mem_norm(mem), m

    def decode(self, dec_in, mem, m):
        L = dec_in.size(1)
        x = self.tok_emb(dec_in) + self.pos_emb(torch.arange(L, device=dec_in.device))
        causal = nn.Transformer.generate_square_subsequent_mask(L, device=dec_in.device)
        h = self.decoder(x, mem, tgt_mask=causal, tgt_is_causal=True, memory_key_padding_mask=m)
        return F.linear(self.dec_norm(h), self.tok_emb.weight)

    def forward(self, b):
        # autocast lives here because DataParallel replicas run in separate threads
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.cfg.fp16):
            mem, m = self.encode(b["pixel"], b["ids"], b["att"], b["tt"], b["boxes"])
            dec = b["dec"]; logits = self.decode(dec[:, :-1], mem, m)
        tgt = dec[:, 1:].masked_fill(dec[:, 1:] == self.pad_id, -100)
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), tgt.reshape(-1), ignore_index=-100)

    @torch.no_grad()
    def generate(self, b, bos, eos):
        mem, m = self.encode(b["pixel"], b["ids"], b["att"], b["tt"], b["boxes"])
        B = mem.size(0)
        seq = torch.full((B, 1), bos, device=mem.device)
        done = torch.zeros(B, dtype=torch.bool, device=mem.device)
        for _ in range(self.cfg.max_ans_len):
            nxt = self.decode(seq, mem, m)[:, -1].argmax(-1)
            nxt = nxt.masked_fill(done, self.pad_id); seq = torch.cat([seq, nxt[:, None]], 1)
            done |= nxt == eos
            if done.all(): break
        return seq[:, 1:]

In [28]:
def levenshtein(a, b):
    if len(a) < len(b): a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

def anls_single(pred, gts, tau=0.5):
    best = 0.0
    for g in gts:
        p, g = pred.strip().lower(), g.strip().lower()
        nl = levenshtein(p, g) / max(len(p), len(g), 1)
        best = max(best, 1 - nl if nl < tau else 0.0)
    return best

def _num(s):
    try: return float(s.replace(",", "").replace("%", "").replace("$", "").strip())
    except Exception: return None

def relaxed_acc_single(pred, gts, tol=0.05):
    for g in gts:
        pn, gn = _num(pred), _num(g)
        if gn is not None and pn is not None:
            if (gn == 0 and pn == 0) or (gn != 0 and abs(pn - gn) / abs(gn) <= tol): return 1.0
        elif pred.strip().lower() == g.strip().lower(): return 1.0
    return 0.0

def bootstrap_diff(scores_a, scores_b, n=1000, seed=0):
    a, b = np.array(scores_a), np.array(scores_b); rng = np.random.default_rng(seed)
    d = [(a[i] - b[i]).mean() for i in (rng.integers(0, len(a), len(a)) for _ in range(n))]
    return float((a - b).mean()), (float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)))

In [29]:
def _device(): return torch.device("cuda" if torch.cuda.is_available() else "cpu")
def _to(b, dev): return {k: (v.to(dev) if torch.is_tensor(v) else v) for k, v in b.items()}

@torch.no_grad()
def evaluate(model, hf_split, tok, cfg, max_n=None, chart=False):
    dev = _device(); model.eval(); model.pad_id = tok.pad_token_id
    if max_n and len(hf_split) > max_n: hf_split = hf_split.shuffle(seed=0).select(range(max_n))
    dl = DataLoader(DS(hf_split), batch_size=cfg.batch_size*2, shuffle=False,
                    collate_fn=Collator(tok, cfg, False), num_workers=cfg.num_workers)
    preds, golds = [], []
    for b in dl:
        b = _to(b, dev)
        with torch.autocast(dev.type, dtype=torch.float16, enabled=cfg.fp16 and dev.type == "cuda"):
            out = model.generate(b, tok.cls_token_id, tok.sep_token_id)   # runs on GPU 0 only
        for row in out.tolist():
            row = row[:row.index(tok.sep_token_id)] if tok.sep_token_id in row else row
            preds.append(tok.decode(row, skip_special_tokens=True).strip())
        golds += b["answers"]
    res = {"anls_scores": [anls_single(p, [g]) for p, g in zip(preds, golds)]}
    res["anls"] = float(np.mean(res["anls_scores"]))
    if chart:
        res["ra_scores"] = [relaxed_acc_single(p, [g]) for p, g in zip(preds, golds)]
        res["ra"] = float(np.mean(res["ra_scores"]))
    try:
        import sacrebleu
        res["bleu4"] = sacrebleu.corpus_bleu([p.lower() for p in preds], [[g.lower() for g in golds]],
                                             smooth_method="exp").score
    except ImportError: pass
    res["preds"] = preds[:50]
    return res

def run_experiment(dsdict, name, variant, cfg=None, chart=None):
    cfg = cfg or Cfg(); chart = name.lower().startswith("chart") if chart is None else chart
    random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)
    dev = _device(); os.makedirs(cfg.out_dir, exist_ok=True)
    tok = AutoTokenizer.from_pretrained(cfg.bert_name, use_fast=True)
    val_key = "validation" if "validation" in dsdict else "val"
    model = LGOVF(variant, cfg).to(dev); model.pad_id = tok.pad_token_id
    net = nn.DataParallel(model) if torch.cuda.device_count() > 1 else model
    print(f"[{name}/{variant}] training on {max(1, torch.cuda.device_count())} GPU(s)")

    groups = {}
    for n, p in model.named_parameters():
        pre = n.startswith(("vit.", "bert."))
        nd = n.endswith("bias") or "norm" in n.lower() or "LayerNorm" in n
        groups.setdefault((pre, nd), []).append(p)
    opt = torch.optim.AdamW([{"params": ps, "lr": cfg.lr_pre if pre else cfg.lr_new,
                              "weight_decay": 0.0 if nd else cfg.weight_decay}
                             for (pre, nd), ps in groups.items()], betas=(0.9, 0.999), eps=1e-8)
    dl = DataLoader(DS(dsdict["train"]), batch_size=cfg.batch_size, shuffle=True, drop_last=True,
                    collate_fn=Collator(tok, cfg, True), num_workers=cfg.num_workers, pin_memory=True)
    steps = cfg.epochs * (len(dl) // cfg.grad_accum)
    sched = get_linear_schedule_with_warmup(opt, int(cfg.warmup_frac * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.fp16 and dev.type == "cuda")
    ckpt = f"{cfg.out_dir}/{name}_{variant}.pt"; best, bad, hist = -1.0, 0, []

    for ep in range(cfg.epochs):
        model.train(); tot, n = 0.0, 0; opt.zero_grad(set_to_none=True)
        for i, b in enumerate(dl, 1):
            b = _to(b, dev)
            bt = {k: v for k, v in b.items() if k != "answers"}   # strings can't be scattered to GPUs
            loss = net(bt).mean() / cfg.grad_accum                # .mean() merges per-GPU losses
            scaler.scale(loss).backward(); tot += loss.item() * cfg.grad_accum; n += 1
            if i % cfg.grad_accum == 0:
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), cfg.clip)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
        v = evaluate(model, dsdict[val_key], tok, cfg, cfg.max_val, chart)
        hist.append(dict(epoch=ep+1, loss=tot/n, val_anls=v["anls"], val_ra=v.get("ra")))
        print(f"[{name}/{variant}] ep{ep+1} loss {tot/n:.4f} val ANLS {v['anls']:.4f}" +
              (f" RA {v['ra']:.4f}" if chart else ""))
        if v["anls"] > best: best, bad = v["anls"], 0; torch.save(model.state_dict(), ckpt)
        else:
            bad += 1
            if bad >= cfg.patience: print("early stop"); break

    model.load_state_dict(torch.load(ckpt, map_location=dev))
    test = evaluate(model, dsdict["test"], tok, cfg, None, chart)
    print(f"[{name}/{variant}] TEST ANLS {test['anls']:.4f}" + (f" RA {test['ra']:.4f}" if chart else "") +
          (f" BLEU-4 {test['bleu4']:.2f}" if "bleu4" in test else ""))
    json.dump({"cfg": asdict(cfg), "history": hist, "test": test},
              open(f"{cfg.out_dir}/{name}_{variant}_results.json", "w"))
    return dict(model=model, test=test, history=hist, ckpt=ckpt)

In [30]:
small = Cfg(epochs=1, max_val=200)
tiny_s = {k: v.select(range(min(200, len(v)))) for k, v in slides.items()}
_ = run_experiment(tiny_s, "slides_trial", "v5", small)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

... (6 outputs left out)

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.pooler.dense.weight                   | UNEXPECTED |  | 
bert.pooler.dense.bias                     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[slides_trial/v5] training on 2 GPU(s)


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


[slides_trial/v5] ep1 loss 6.2088 val ANLS 0.0000
[slides_trial/v5] TEST ANLS 0.0000 BLEU-4 0.03


In [33]:
!rm -f /kaggle/working/ckpt/slides_trial_v5.pt /kaggle/working/ckpt/slides_trial_v5_results.json
!pip cache purge
!apt-get clean

Files removed: 18 (4.6 MB)
Directories removed: 44


In [34]:
!du -sh /kaggle/working/* /kaggle/temp/* /tmp /root/.cache/huggingface/* 2>/dev/null | sort -h | tail -12
!ls -la /kaggle/working/ckpt
try: print(len(chart["train"]), len(slides["train"]))
except NameError as e: print("variables lost:", e)

4.0K	/kaggle/working/ckpt
12K	/tmp
1.1M	/root/.cache/huggingface/xet
1.4G	/kaggle/working/chart_ocr
2.7G	/root/.cache/huggingface/hub
13G	/kaggle/working/slides_ocr
51G	/root/.cache/huggingface/datasets
total 8
drwxr-xr-x 2 root root 4096 Nov  7 10:00 .
drwxr-xr-x 6 root root 4096 Nov  7 09:59 ..
28066 23939


In [36]:
from datasets import load_from_disk
import gc, shutil, os

# 1. reload from the saved copies (memory-mapped, low RAM)
chart2  = load_from_disk("/kaggle/working/chart_ocr")
slides2 = load_from_disk("/kaggle/working/slides_ocr")

# 2. sanity check before touching anything
for name, d in [("chart", chart2), ("slides", slides2)]:
    print(name, {k: len(v) for k, v in d.items()}, d["train"].column_names)
ex = chart2["train"][0]
print(len(ex["words"]), ex["boxes"][:2], ex["question"], "|", ex["answer"])
assert len(chart2["train"]) == 28066 and len(slides2["train"]) == 23939

Loading dataset from disk:   0%|          | 0/21 [00:00<?, ?it/s]

chart {'train': 28066, 'val': 1904, 'test': 2490} ['id', 'question', 'answer', 'image', 'source', 'subset', 'words', 'boxes']
slides {'train': 23939, 'val': 2801, 'test': 3288} ['id', 'question', 'answer', 'image', 'deck_name', 'image_url', 'source', 'words', 'boxes']
49 [[946, 37, 961, 50], [106, 63, 147, 72]] As of 2021, how many championship titles had Ferrari won? | 16


In [37]:
# switch to the reloaded versions and drop everything else
chart, slides = chart2, slides2
for v in ["dataset", "chartqa", "chart2", "slides2", "_", "r", "res_c", "res_s"]:
    globals().pop(v, None)
gc.collect(); torch.cuda.empty_cache()

# delete the stale cache (hub/ with the BERT and ViT weights is a different folder and stays)
shutil.rmtree("/root/.cache/huggingface/datasets", ignore_errors=True)
os.makedirs("/root/.cache/huggingface/datasets", exist_ok=True)

print(torch.cuda.memory_allocated(0) / 1e9, "GB held on GPU 0")
!df -h /kaggle/working /root | cat

0.019136512 GB held on GPU 0
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G   14G  6.0G  70% /kaggle/working
overlay         8.0T  7.1T  952G  89% /


In [38]:
!du -sh /root/.cache/huggingface/datasets

4.0K	/root/.cache/huggingface/datasets


In [39]:
cfg = Cfg(epochs=3, patience=2)

res_c = run_experiment(chart,  "chartqa", "v5", cfg)   # ANLS + Relaxed Accuracy
res_s = run_experiment(slides, "slides",  "v5", cfg)   # ANLS

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.bias   | UNEXPECTED |  | 
pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.pooler.dense.weight                   | UNEXPECTED |  | 
bert.pooler.dense.bias                     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[chartqa/v5] training on 2 GPU(s)
[chartqa/v5] ep1 loss 3.5265 val ANLS 0.0894 RA 0.1227
[chartqa/v5] ep2 loss 2.6136 val ANLS 0.1217 RA 0.1260
[chartqa/v5] ep3 loss 2.3118 val ANLS 0.1490 RA 0.1447
[chartqa/v5] TEST ANLS 0.1368 RA 0.1390 BLEU-4 0.75


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.bias   | UNEXPECTED |  | 
pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.pooler.dense.weight                   | UNEXPECTED |  | 
bert.pooler.dense.bias                     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[slides/v5] training on 2 GPU(s)
[slides/v5] ep1 loss 2.6471 val ANLS 0.0379
[slides/v5] ep2 loss 1.8010 val ANLS 0.0493
[slides/v5] ep3 loss 1.5737 val ANLS 0.0634
[slides/v5] TEST ANLS 0.0000 BLEU-4 0.02


In [41]:
cfg = Cfg(epochs=3, patience=2)          # same settings as the V5 run

res_c2 = run_experiment(chart,  "chartqa", "v2", cfg)   # ANLS + Relaxed Accuracy
res_s2 = run_experiment(slides, "slides",  "v2", cfg)   # ANLS

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.bias   | UNEXPECTED |  | 
pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.pooler.dense.weight                   | UNEXPECTED |  | 
bert.pooler.dense.bias                     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[chartqa/v2] training on 2 GPU(s)
[chartqa/v2] ep1 loss 3.5688 val ANLS 0.0754 RA 0.1093
[chartqa/v2] ep2 loss 2.6614 val ANLS 0.1212 RA 0.1173
[chartqa/v2] ep3 loss 2.3734 val ANLS 0.1377 RA 0.1367
[chartqa/v2] TEST ANLS 0.1252 RA 0.1357 BLEU-4 0.50


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.bias   | UNEXPECTED |  | 
pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.pooler.dense.weight                   | UNEXPECTED |  | 
bert.pooler.dense.bias                     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[slides/v2] training on 2 GPU(s)
[slides/v2] ep1 loss 2.6684 val ANLS 0.0407
[slides/v2] ep2 loss 1.8079 val ANLS 0.0552
[slides/v2] ep3 loss 1.5921 val ANLS 0.0625
[slides/v2] TEST ANLS 0.0000 BLEU-4 0.02


In [44]:
import os, shutil, zipfile

src = "/kaggle/working/ckpt"
size = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(src) for f in fs)
free = shutil.disk_usage("/kaggle/working").free
print(f"ckpt folder: {size/1e9:.2f} GB | free space: {free/1e9:.2f} GB")
for f in sorted(os.listdir(src)): print(f"  {f}  {os.path.getsize(os.path.join(src, f))/1e9:.2f} GB")

if size * 1.05 < free:
    with zipfile.ZipFile("/kaggle/working/ckpt.zip", "w", zipfile.ZIP_STORED) as z:   # STORED = no compression, fast
        for f in os.listdir(src): z.write(os.path.join(src, f), f)
    print("created /kaggle/working/ckpt.zip", round(os.path.getsize("/kaggle/working/ckpt.zip")/1e9, 2), "GB")
else:
    print("NOT ENOUGH SPACE - zip skipped. Download the files one by one instead (see below).")

ckpt folder: 3.98 GB | free space: 2.45 GB
  chartqa_v2.pt  0.99 GB
  chartqa_v2_results.json  0.00 GB
  chartqa_v5.pt  0.99 GB
  chartqa_v5_results.json  0.00 GB
  slides_v2.pt  0.99 GB
  slides_v2_results.json  0.00 GB
  slides_v5.pt  0.99 GB
  slides_v5_results.json  0.00 GB
NOT ENOUGH SPACE - zip skipped. Download the files one by one instead (see below).


In [47]:
import os, glob

CONFIRM = True       # change to True to actually delete

keep, delete = [], []
for f in sorted(glob.glob("/kaggle/working/ckpt/*")):
    name = os.path.basename(f)
    is_v5 = "_v5" in name and "trial" not in name       # keeps chartqa_v5.*, slides_v5.*
    (keep if is_v5 else delete).append(f)

print("KEEP:")
for f in keep: print(f"  {os.path.basename(f)}  {os.path.getsize(f)/1e9:.2f} GB")
print("DELETE:")
for f in delete: print(f"  {os.path.basename(f)}  {os.path.getsize(f)/1e9:.2f} GB")

if CONFIRM:
    for f in delete: os.remove(f)
    print("deleted", len(delete), "files")
else:
    print("\nDry run only. Set CONFIRM = True and run again to delete.")

KEEP:
  chartqa_v5.pt  0.99 GB
  chartqa_v5_results.json  0.00 GB
  slides_v5.pt  0.99 GB
  slides_v5_results.json  0.00 GB
DELETE:
  chartqa_v2.pt  0.99 GB
  chartqa_v2_results.json  0.00 GB
  slides_v2.pt  0.99 GB
  slides_v2_results.json  0.00 GB
deleted 4 files


In [ ]:
import gc, torch, warnings
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()
warnings.filterwarnings("ignore", message=".*gather along dimension 0.*")

gc.collect(); torch.cuda.empty_cache()
print([round(torch.cuda.memory_allocated(i) / 1e9, 2) for i in range(2)], "GB held per GPU (should be ~0)")

cfg10 = Cfg(epochs=10, patience=3, batch_size=8, grad_accum=4,    # 8 x 4 = effective 32, same as before
            out_dir="/kaggle/working/ckpt_10ep")

res_c10 = run_experiment(chart, "chartqa", "v5", cfg10)

print("10 epochs: ANLS", res_c10["test"]["anls"], "RA", res_c10["test"]["ra"])
print(res_c10["history"])

# compare with the 3-epoch V5 run (res_c is kept in memory)
m, ci = bootstrap_diff(res_c10["test"]["ra_scores"], res_c["test"]["ra_scores"])
print(f"RA 10ep - 3ep: {m:+.4f}  95% CI ({ci[0]:+.4f}, {ci[1]:+.4f})")

[2.03, 0.02] GB held per GPU (should be ~0)
[chartqa/v5] training on 2 GPU(s)
[chartqa/v5] ep1 loss 3.8268 val ANLS 0.0802 RA 0.1093
[chartqa/v5] ep2 loss 2.7179 val ANLS 0.1130 RA 0.1253
[chartqa/v5] ep3 loss 2.3677 val ANLS 0.1527 RA 0.1447
[chartqa/v5] ep4 loss 2.0759 val ANLS 0.1769 RA 0.1507
[chartqa/v5] ep5 loss 1.8244 val ANLS 0.1781 RA 0.1567
